### Tensor
similar to List but an Object and Helps in Keeping Track. Just similar to torch.Tensor at core

In [267]:
import numpy as np
class Tensor:
    def __init__(self, data, children=(), op='', label=''):
        self.data = np.atleast_2d(np.asarray(data))
        self.label = label
        self.grad = np.asarray(np.zeros(shape=(self.data.shape)))
        #Internal variable for in
        self._op = op
        self._prev = children
        self._backward = lambda:None

    def __repr__(self):
        return f"Tensor :{self.data}"

    def __mul__(self, other):
        try:
            if not isinstance(other, Tensor):
                other = Tensor(other)

            if other.data.shape == (1, 1) and self.data.shape[1] != other.data.shape[0]:
                return self.elementwise_mul(other)


            # print("dims Self",self.data.shape,"dims other: ", type(other))
            out = Tensor(
                data=self.data @ other.data,
                children=(self, other),
                op='*'
            )
            def _backward():
                self.grad += out.grad @ other.data.T
                other.grad += self.data.T @ out.grad

            out._backward = _backward
            return out
        except Exception as e:
            raise ValueError(
                f"Matrix multiplication failed: self dimensions {self.data.shape}, "
                f"other dimensions {other.data.shape}"
            ) from e

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Tensor(np.pow(self.data,other), children=(self,), op=f'**{other}')
        def _backward():
            self.grad += (other * np.pow(self.data,other-1)) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        out = Tensor(np.exp(self.data),children=(self,), op="exp")

        def _backward():
            self.grad += out.data * out.grad  #element wise multiplication
        out._backward = _backward
        return out

    def transpose(self):
        out = Tensor(np.array(self.data).transpose(),op='T',children=(self,))
        def _backward():
            self.grad += out.grad.transpose()
        out._backward = _backward
        return out

    def softmax(self) -> 'Tensor':
        shifted = self.data - np.max(self.data, axis=1, keepdims=True)
        exp_shifted = np.exp(shifted)
        softmax_data = exp_shifted / np.sum(exp_shifted, axis=1, keepdims=True)

        out = Tensor(softmax_data, op='softmax', children=(self,))

        def _backward():
            # correct softmax Jacobian, per row:
            # dL/dx_i = s_i * (dL/dout_i - sum_j(s_j * dL/dout_j))
            s = out.data
            dot = np.sum(s * out.grad, axis=1, keepdims=True)
            self.grad += s * (out.grad - dot)

        out._backward = _backward
        return out

    def relu(self):
        b = self.data
        out = Tensor(np.where(np.greater_equal(0, b), 0, b),children=(self,),label='ReLU')

        def _backward():
            self.grad += (out.data>0)*out.grad

        out._backward = _backward
        return out

    def backward(self):
        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)


        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = np.asarray(np.ones(shape=(self.data.shape)))

        for v in reversed(topo):
            v._backward()

    def __sub__(self, other): # self - other
        return self + (-other)

    def __neg__(self): # -self
        return self * -1

    def __rmul__(self, other): #__rmul__ is used by python when a * b can't be calculated then python tries b * a
        return self*other

    def __radd__(self, other): # other + self
        return self + other

    def cat(tensors:list['Tensor'], axis = 1):
        if not isinstance(tensors, (list, tuple)):
            raise TypeError("cat() expects a list or tuple of Tensor objects")
        if len(tensors) == 0:
            raise ValueError("cat() requires at least one tensor")
        if not all(isinstance(t, Tensor) for t in tensors):
            raise TypeError("All elements in tensors must be Tensor instances")
        if len(tensors)==1: return tensors[0]

        out_data = np.concatenate([t.data for t in tensors],axis=axis)

        out = Tensor(out_data,
           children=tuple(tensors),
            op="cat"
        )
        def _backward():
            start = 0
            for t in tensors:
                size = t.grad.shape[axis]
                end = start + size
                idx = [slice(None)] * out.grad.ndim
                idx[axis] = slice(start,end)
                t.grad += out.grad[tuple(idx)]
                start = end
        out._backward = _backward
        return out

    def unbroadcast(self,grad, shape, label=''):
        # Remove extra dimensions on the left
        while grad.ndim > len(shape):
            grad = np.sum(grad, axis=0)
        # Sum dimensions where original shape was 1
        for axis, size in enumerate(shape):
            if size == 1 and grad.shape[axis] != 1:
                grad = np.sum(grad, axis=axis, keepdims=True)
        return grad

    def __add__(self, other):
        if not isinstance(other, Tensor):
            other = Tensor(other)
        # print("add: ", self, other)
        out = Tensor(np.add(self.data, other.data), children=(self, other), op='+')
        def _backward():
            # print("trying to add: ", other.grad, out.data.shape)
            self.grad += self.unbroadcast(out.grad, self.data.shape)
            other.grad += self.unbroadcast(out.grad, other.data.shape)
        out._backward = _backward

        return out

    def elementwise_mul(self, other):
        if not isinstance(other, Tensor):
            other = Tensor(other)
        out = Tensor(
            data=self.data * other.data,
            children=(self, other),
            op='ew*'
        )
        def _backward():
            self.grad +=  self.unbroadcast(other.data * out.grad, shape=self.data.shape)
            other.grad += self.unbroadcast(self.data * out.grad, shape=other.data.shape)
        out._backward = _backward
        return out

    def mean(self,axis=1):
        out = Tensor(np.mean(self.data, axis=axis,keepdims=True),children=(self,),op='mean')
        def _backward():
            if axis is None:
                # If mean is taken over all elements, gradient is distributed equally
                self.grad += out.grad / self.data.size
            else:
                # If mean is taken over a specific axis
                self.grad += (1/self.data.shape[axis])*out.grad
        out._backward = _backward
        return out

    def std(self,axis = 1):
        out = Tensor(np.std(self.data,axis=axis,keepdims=True),children=(self,),op='std')
        def _backward():
            self.grad = out.grad * (self.data - np.mean(self.data)) / (self.data.shape[axis] * out.data)
        out._backward = _backward
        return out

    def tanh(self):
        e = self.elementwise_mul(2).exp()
        out = Tensor(
            (e - 1).data * ((e + 1)**-1).data,
            children=(self,),
            op='tanh'
        )

        def _backward():
            self.grad += out.grad * (1 - out.data**2)

        out._backward = _backward
        return out

### Graphvis for visualisation

In [268]:
%%script false --no-raise-error

!pip install graphviz
from graphviz import Digraph
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir})

    def matrix_html(matrix):
        matrix = np.atleast_2d(matrix)
        rows = []
        for row in matrix:
            cells = ''.join(
                f'<TD BORDER="1" CELLPADDING="4">{value:.4g}</TD>'
                for value in row
            )
            rows.append(f'<TR>{cells}</TR>')
        return (
            '<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0">'
            + ''.join(rows) +
            '</TABLE>'
        )

    for n in nodes:
        label = (
            f'<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0">'
            f'<TR><TD><B>{n.label}</B></TD></TR>'
            f'<TR><TD>{matrix_html(n.data)}</TD></TR>'
            f'<TR><TD>grad</TD></TR>'
            f'<TR><TD>{matrix_html(n.grad)}</TD></TR>'
            f'</TABLE>>'
        )

        dot.node(
            name=str(id(n)),
            label=label,
            shape='plain'
        )

        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

def draw_graph(end_targate):
    end_targate.backward()
    dot = draw_dot(end_targate)
    dot.body[:] = [
        line.replace("<B></B>", "<B>&#160;</B>")
        for line in dot.body
    ]
    dot.render()
    display(dot)


### Multi layer perceptron
created Neuron --> Layer --> MLP

### Gradient Descent

In [269]:
def GradientDescent(model,x,y,epoch=200,lr=0.1):
    lostHistory=[]
    ypred:list = model(x) #gives a list of outputs
    ypred:Tensor = Tensor.cat(ypred, axis=1) #list of Q Tensors(mx1) to be converted back to Tensor shape(mxQ) rows to be contatinate
    if not isinstance(y,Tensor):
        y = Tensor(y,label='Output').transpose()
    if y.data.shape != ypred.data.shape:
        msg = (
            "prediction and true output shape mismatch: "
            f"y.shape={y.data.shape}, ypred.shape={ypred.data.shape}"
        )
        raise ValueError(msg)

    for _ in range(epoch):
        # print(np.shape(y.data))
        ypred = model(x) #gives a list of outputs
        ypred = Tensor.cat(ypred, axis=1) #list to be converted back to Tensors
        loss = ((ypred - y) ** 2).mean(axis=1)

        for p in model.parameters():
            p.grad = 0.0

        loss.backward()

        for p in model.parameters():
            p.data -= lr * p.grad
        lostHistory.append(loss.data)
        # print("loss:", loss.data[0])
    return ypred, lostHistory

In [270]:
from typing import Literal
import random

class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

# Single Neuron with its own weights and bias based on number of inputs(n_inp)
class Neuron(Module):
    def __init__(self, n_inp, non_linear=True, activation_fn: Literal["tanh", "relu"] = "tanh"):
        valid_activations = {"tanh", "relu"}

        if activation_fn not in valid_activations:
            raise ValueError(
                f"activation_fn must be one of {valid_activations}"
            )

        self.W = Tensor(
            [[np.random.uniform(-1, 1) for _ in range(n_inp)]],
            label="w"
        )
        self.b = Tensor([[0.0]], label="bias")
        self.non_linear = non_linear
        self.activation_fn = activation_fn

    def __call__(self, x):
        if isinstance(x, list) and all(isinstance(xi, Tensor) for xi in x):
            x = Tensor.cat(x, axis=1).transpose()   # previous layer's outputs -> (n_inp, 1)
        elif isinstance(x,Tensor):
            x = x.transpose()
        else:
            x = Tensor(np.atleast_2d(x).transpose())   # raw numeric input -> (n_inp, 1)
        act = (self.W * x) + self.b
        act = act.transpose()
        if not self.non_linear: #Linear actication
            return act
        activation = getattr(act, self.activation_fn) #Non Linear actication
        return activation()

    def parameters(self):
        return [self.W, self.b]

    def __repr__(self):
        return f"{'Tanh' if self.non_linear else 'Linear'} Neuron({self.W.data.shape[1]})"


# Layer contains
class Layer(Module):
    def __init__(self,n_inp, n_out,activation_fn: Literal["tanh", "relu"] = "tanh", non_linear=True, **kwargs):
        #number of neuron depends on number Outputs we want
        #each neuron's weights depends on number of inputs it recienve's
        self.neurons:list[Neuron] = [Neuron(n_inp,activation_fn=activation_fn ,**kwargs) for _ in range(n_out)]

    def __call__(self, x):

        out:list[Tensor] = [n(x) for n in self.neurons]

        return out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):
    def __init__(self, n_inp:int, Layer_outs:list[int],activation_fn: Literal["tanh", "relu"] = "tanh"):
        input_size = [n_inp] + Layer_outs
        self.layers:list[Layer] = [
            Layer(
                n_inp=input_size[out],
                n_out=input_size[out + 1],
                non_linear=(out != len(Layer_outs) - 1),
                activation_fn=activation_fn
            )
            for out in range(len(Layer_outs))
        ]

    def __call__(self, x):
        for layer in self.layers:

            x = layer(x)

        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def Fit(self,x,y,epoch=200,lr=0.1, batch_size=32):
        lostHistory=[]
        ypred:list = self(x) #gives a list of outputs
        ypred:Tensor = Tensor.cat(ypred, axis=1) #list of Q Tensors(mx1) to be converted back to Tensor shape(mxQ) rows to be contatinate
        if not isinstance(y,Tensor):
            y = Tensor(y,label='Output')
        if y.data.shape != ypred.data.shape:
            msg = (
                "prediction and true output shape mismatch: "
                f"y.shape={y.data.shape}, ypred.shape={ypred.data.shape}"
            )
            raise ValueError(msg)
    
        for _ in range(epoch):
            # print(np.shape(y.data))
            ypred = self(x) #gives a list of outputs
            ypred = Tensor.cat(ypred, axis=1) #list to be converted back to Tensors
            loss = ((ypred - y) ** 2).mean(axis=1)
    
            for p in self.parameters():
                p.grad = 0.0
    
            loss.backward()
    
            for p in self.parameters():
                p.data -= lr * p.grad
            lostHistory.append(loss.data)
            # print("loss:", loss.data[0])
        return ypred, lostHistory

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

### Transformer

In [271]:
import numpy as np
# from .backprop import Tensor

def sinusoidal_positional_encoding(m_words:int, n_dims:int):
    #number of words, n_dims = embedding dims of each word
    #P_E(pos,2i) = sin(pos/10000^(2i/n_dims))
    #P_E(pos,2i+1) = cos(pos/10000^(2i/n_dims))

    #division term = position/10000^(2i/n_dims) = position * 10000^-(2i/n_dims) = position * e^(-2*position*log(10000)/n_dims)
    position = np.arange(m_words)[:,np.newaxis]
    div_term = np.exp(-2*position*np.log(10000)/n_dims)

    pos_embed = np.zeros(shape=(m_words,n_dims))
    #even position
    pos_embed[:,0::2] = np.sin(position*div_term)
    pos_embed[:,1::2] = np.cos(position*div_term)
    pos_embed = Tensor(pos_embed,label='pos_embed')
    return pos_embed

def layer_normalize(matrix:list):
    # matrix = multihead attention embedding [mxn]
    constant = 0.001 # to preven from being zero at denominator
    l2 = np.array(matrix)
    mean = np.mean(l2,axis=1,keepdims=True)[:,np.newaxis] #returns [mX1]
    x_minus_mean = l2 - mean
    var = np.sqrt(np.mean(np.square(x_minus_mean),axis=1)[:,np.newaxis] + constant) #[mx1]

    normalized = x_minus_mean/var
    return normalized

class attention_head:
    def __init__(self, shape: tuple[int, int] = (1, 1)):
        self.m, self.n = shape
        self.W_q = Tensor(np.random.randn(self.n,self.n),label='w_q')
        self.W_k = Tensor(np.random.randn(self.n,self.n),label='w_k')
        self.W_v = Tensor(np.random.randn(self.n,self.n),label='w_v')

    def __call__(self, P_E:Tensor):
        # """Attention"""
        Q = P_E * self.W_q
        K = P_E * self.W_k
        V = P_E * self.W_v
        div_factor = Tensor(1/np.sqrt(self.n))
        similarity = div_factor.elementwise_mul(Q * K.transpose())
        context_embed:Tensor = (similarity * V).softmax() # softmax((1/sqrt(dk))*Q.K)*V

        # """Add and Norm"""
        context_embed = context_embed + P_E
        return context_embed
    def perameter(self):
        return self.W_q,self.W_k,self.W_v


class Transformer:
    def __init__(self,word_embeds:list, n_heads:int = 8):
        self.word_embeds = Tensor(word_embeds,label='Data ingestion')
        print("Data Ingestion Successfull, Shape: ", self.word_embeds.data.shape)
        self.m,self.n = np.shape(self.word_embeds.data)
        self.n_heads = n_heads #number of attention head
        self.W_multi_head = Tensor(np.random.randn(self.n*n_heads, self.n),label="multihead")
        self.all_attention_heads = [attention_head(shape=(self.m,self.n)) for _ in range(self.n_heads)]
        self.lr = 0.01
        # """ FFN Model """
        self.model = MLP(n_inp=self.n, Layer_outs=[64, 64, 32, self.n],activation_fn='tanh')

    def encoder(self):
        pos_encode:Tensor = sinusoidal_positional_encoding(self.m,self.n)
        P_E = self.word_embeds+pos_encode  #P_E = positional encoding

        # Multi head attention

        tensors = [head(P_E) for head in self.all_attention_heads]
        multi_head_attention:Tensor = Tensor.cat(tensors) * self.W_multi_head
        add = multi_head_attention + P_E
        add = add -add.mean()
        add_n_norm = add.elementwise_mul(add.std()**-1); add_n_norm.label = 'add n norm'


        # """ Feed Forward Neural Network """
        model = self.model
        FFNN1:list[Tensor] = model(add_n_norm)

        add_n_norm2 = add_n_norm + Tensor.cat(FFNN1)
        return add_n_norm2


    def Perameter(self):
        params = [self.W_multi_head]
        for head in self.all_attention_heads:
            params.extend(head.perameter())
        params.extend(self.model.parameters())
        return params

In [272]:

word_embeds = [
    [0.6972834, 0.08725786, -0.1506938, -0.67638519],
    [-1.30838017, -1.7098192, -1.44106524, -1.63344123],
]
word_embeds_id = [[1,1,1,1],[1,1,1,1]]
transformer = Transformer(word_embeds)
encoder = transformer.encoder()
model  =  transformer.model.Fit(word_embeds,word_embeds_id,epoch=10,batch_size = 1)
model


Data Ingestion Successfull, Shape:  (2, 4)


(Tensor :[[0.99910878 0.94246483 0.9999057  0.99931449]
  [0.99591914 0.99965882 0.99992641 0.99904254]],
 [array([[1.20396577],
         [1.17713579]]),
  array([[0.7754243],
         [1.0699082]]),
  array([[1.06313483],
         [0.28514392]]),
  array([[9.94314491e-01],
         [8.91226917e-05]]),
  array([[9.91409686e-01],
         [8.38110882e-05]]),
  array([[9.84072923e-01],
         [7.67963892e-05]]),
  array([[9.50062272e-01],
         [6.58909451e-05]]),
  array([[2.45234176e-01],
         [4.21056882e-05]]),
  array([[9.73099677e-04],
         [4.41557154e-06]]),
  array([[8.27892262e-04],
         [4.42298197e-06]])])

# Data Prep

In [273]:
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

In [274]:
#Read csv-------------------
%%script false --no-raise-error
df = pd.read_csv('../database/Reddit_Data.csv')
df.head(1)

UsageError: Line magic function `%%script` not found.


#### Preprocessing

In [ ]:
%%script false --no-raise-error

print(df.info())
print("Target data distribution: ",[int((i/37249)*100) for i in df.category.value_counts()])


df.dropna(inplace=True)
print("Total null Values: ",df.isnull().sum())
#lower casing
df['clean_comment'] = df['clean_comment'].str.lower()

#remove punctuation
import string
exclude = string.punctuation
print(exclude)
def removepunc(text:str):
    return text.translate(str.maketrans('','',exclude))

df['clean_comment'] = df['clean_comment'].apply(removepunc)
df.head()

#stop word removal
!pip install nltk
import nltk
from nltk.corpus import stopwords
import time
nltk.download('stopwords')
en_stopwords = stopwords.words('english')

def removestopwords(text:str):
    newword = []
    for word in text.split():
        if word not in en_stopwords:
            newword.append(word)
    x = newword[:]
    newword.clear()
    return " ".join(x)

df['clean_comment'] = df.clean_comment.apply(removestopwords)

#emoji replacement with their meaning
!pip install emoji
import emoji
df.clean_comment = df.clean_comment.apply(emoji.demojize)


#Nltk not that accurate but for now, just use it
from nltk.tokenize import word_tokenize
nltk.download('punkt_tab')
df.clean_comment = df.clean_comment.apply(word_tokenize)
df.head()

from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
def stem_word(text_row):
    return (" ".join(ps.stem(word) for word in text_row))
df.clean_comment = df.clean_comment.apply(stem_word)
df.head(1)



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37249 entries, 0 to 37248
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   clean_comment  37149 non-null  object
 1   category       37249 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 582.1+ KB
None
Target data distribution:  [42, 35, 22]
Total null Values:  clean_comment    0
category         0
dtype: int64
!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,clean_comment,category
0,famili mormon never tri explain still stare pu...,1


#### Vocab Preparing for bert like Model Training

In [ ]:
%%script false --no-raise-error
df['split_comment'] = df.clean_comment.str.split()
df.head()

,clean_comment,category,split_comment
0,famili mormon never tri explain still stare pu...,1,"[famili, mormon, never, tri, explain, still, s..."
1,buddhism much lot compat christian especi cons...,1,"[buddhism, much, lot, compat, christian, espec..."
2,serious say thing first get complex explain no...,-1,"[serious, say, thing, first, get, complex, exp..."
3,learn want teach differ focu goal wrap paper b...,0,"[learn, want, teach, differ, focu, goal, wrap,..."
4,benefit may want read live buddha live christ ...,1,"[benefit, may, want, read, live, buddha, live,..."


In [ ]:
%%script false --no-raise-error
unique_words = set()
for comment_words in df['split_comment']:
    unique_words.update(comment_words)

vocabulary = ['[PAD]', '[CLS]','[UNK]', '[SEP]', '[MASK]'] + list(unique_words)
word_to_id = {word: idx for idx, word in enumerate(vocabulary)}

# Display a sample of the created word_to_id mapping and its size
print(f"Vocabulary size: {word_to_id}")


Vocabulary size: {'[PAD]': 0, '[CLS]': 1, '[UNK]': 2, '[SEP]': 3, '[MASK]': 4, 'fault': 5, '・゜': 6, 'mode': 7, 'extremeley': 8, 'oem': 9, 'finder': 10, '1030pm': 11, 'animationcallback': 12, 'tarrif': 13, 'miglior': 14, 'dest': 15, '2700€': 16, '10': 17, 'b2c': 18, 'ianuyi': 19, 'ud': 20, 'noam': 21, 'cardin': 22, 'bacteri': 23, 'buona': 24, 'motherfuck': 25, 'cocoon': 26, 'alli': 27, 'adcb': 28, 'dugaa': 29, 'bur': 30, 'devolv': 31, 'slight': 32, 'perera': 33, 'potif': 34, 'breitbart': 35, 'mainland': 36, 'premchand': 37, 'rapey': 38, 'metano': 39, 'sandhya': 40, 'dda': 41, 'khairaat': 42, 'bqwrnloqh20': 43, 'warranti': 44, 'thackeray': 45, 'interv': 46, 'boe': 47, 'beetl': 48, 'jill': 49, 'psychoact': 50, '70r': 51, 'izzlem': 52, 'zoom': 53, 'scoop': 54, 'artharth': 55, 'kriya': 56, 'bhabhi': 57, 'kiekviena': 58, 'hyperventil': 59, 'faulti': 60, 'bussi': 61, 'helmet': 62, 'download': 63, 'governemnt': 64, 'boom': 65, 'homai': 66, 'esu': 67, 'twitterati': 68, 'spengo': 69, 'ana': 70, 

### Sentence to Vector

### Masking + Length Equilizing

In [ ]:
%%script false --no-raise-error
import random

def mask_input_rows(x: list, mask_prob=0.15):
    # Sentence words to Vector
    #all sentence doesn't have equal length:  It needs equal length: 94% of data las len<64; I'll use it as base trunket greater then this
    word_to_vec = [word_to_id[word] for word in x]
    word_to_vec = word_to_vec[:62]+ [word_to_id['[SEP]']] if len(word_to_vec) > 62 else word_to_vec+ [word_to_id['[SEP]']] + [word_to_id['[PAD]']] * (62 - len(word_to_vec))
    sentence_vec = [word_to_id['[CLS]']] + word_to_vec

    labels = [-100] * len(sentence_vec)  # if its not -100 get the -log(prediction) if the prediction is lower --> high loss else low loss

    for index in range(1, len(sentence_vec) - 1):  # skip CLS (0) and SEP (last)
        if random.random() < mask_prob:
            labels[index] = sentence_vec[index]  # store original id as label

            r = random.random()
            if r < 0.8:
                sentence_vec[index] = word_to_id['[MASK]']
            elif r < 0.9:
                sentence_vec[index] = random.randint(0, len(word_to_id) - 1)
            # else: leave unchanged (10%), label still set above

    return sentence_vec, labels

df[['Masked_Sentence_Vec', 'Labels']] = df['split_comment'].apply(lambda x: pd.Series(mask_input_rows(x)))
df.head()


,clean_comment,category,split_comment,Masked_Sentence_Vec,Labels
0,famili mormon never tri explain still stare pu...,1,"[famili, mormon, never, tri, explain, still, s...","[1, 29758, 24131, 19882, 34180, 4, 26642, 2283...","[-100, -100, -100, -100, -100, 10869, -100, -1..."
1,buddhism much lot compat christian especi cons...,1,"[buddhism, much, lot, compat, christian, espec...","[1, 4176, 30662, 34014, 19967, 569, 39152, 360...","[-100, -100, -100, -100, -100, -100, -100, -10..."
2,serious say thing first get complex explain no...,-1,"[serious, say, thing, first, get, complex, exp...","[1, 4, 34593, 25961, 9335, 4, 31897, 10869, 4,...","[-100, 6358, -100, -100, -100, 6147, -100, -10..."
3,learn want teach differ focu goal wrap paper b...,0,"[learn, want, teach, differ, focu, goal, wrap,...","[1, 11188, 25754, 23749, 4, 5897, 8055, 38883,...","[-100, -100, -100, -100, 6372, -100, -100, -10..."
4,benefit may want read live buddha live christ ...,1,"[benefit, may, want, read, live, buddha, live,...","[1, 8682, 4, 25754, 25633, 32905, 22338, 32905...","[-100, -100, 28177, -100, -100, -100, -100, -1..."


In [ ]:
%%script false --no-raise-error
from sklearn.model_selection import train_test_split
x = df['Masked_Sentence_Vec']
y = df['Labels']
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)


print(x_train.shape)
print("x_train:", x_train.shape)
print("x_test:", x_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)



(29719,)
x_train: (29719,)
x_test: (7430,)
y_train: (29719,)
y_test: (7430,)


### Its time to train

In [ ]:
%%script false --no-raise-error
# Trans = Transformer(x_train)
tr = Transformer(x_train)